# FIAP Threat Model AI

## Treinamento do modelo YOLOv8

Este notebook realiza o treinamento do modelo YOLOv8 para detecção de componentes em diagramas de arquitetura de software.

Etapas:

1. Verificar disponibilidade da GPU
2. Instalar dependências
3. Carregar o dataset
4. Validar a estrutura do dataset
5. Treinar o modelo
6. Avaliar os resultados
7. Baixar o modelo treinado

In [1]:
!nvidia-smi

Mon Jun 29 23:04:35 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Verificação da GPU

O treinamento será realizado utilizando uma GPU Tesla T4 disponibilizada pelo Google Colab.

A utilização de GPU reduz significativamente o tempo de treinamento do modelo.

In [2]:
!pip install -q ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 53.6 MB/s eta 0:00:00


## Instalação das dependências

Nesta etapa é instalada a biblioteca Ultralytics, responsável pelo treinamento do modelo YOLOv8.

In [3]:
from ultralytics import YOLO
from google.colab import files

print("Ultralytics instalado com sucesso!")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics instalado com sucesso!


## Upload do Dataset

O dataset utilizado foi gerado pelo projeto **FIAP Dataset Generator**.

Ele contém:

- imagens PNG
- labels no formato YOLO
- arquivos Draw.io
- arquivo dataset.yaml

In [4]:
uploaded = files.upload()

Saving dataset_v3.zip to dataset_v3.zip


In [5]:
!unzip -q dataset_v3.zip -d /content

In [6]:
!ls /content/dataset_v3

dataset.yaml  drawio  images  labels  metadata.csv  preview


## Validação da Estrutura

Antes do treinamento, verificamos se o dataset foi extraído corretamente e se possui a estrutura esperada pelo YOLO.

In [7]:
!cat /content/dataset_v3/dataset.yaml

path: .
train: images/train
val: images/val

names:
  0: user
  1: internet
  2: firewall
  3: waf
  4: load_balancer
  5: api_gateway
  6: web_server
  7: application_server
  8: microservice
  9: database
  10: cache
  11: queue
  12: object_storage
  13: identity_provider
  14: monitoring


In [9]:
yaml_content = """
path: /content/dataset_v3
train: images/train
val: images/val

names:
  0: user
  1: internet
  2: firewall
  3: waf
  4: load_balancer
  5: api_gateway
  6: web_server
  7: application_server
  8: microservice
  9: database
  10: cache
  11: queue
  12: object_storage
  13: identity_provider
  14: monitoring
"""

with open("/content/dataset_v3/dataset_colab.yaml", "w") as f:
    f.write(yaml_content)

print("dataset_colab.yaml criado com sucesso!")

dataset_colab.yaml criado com sucesso!


In [10]:
!cat /content/dataset_v3/dataset_colab.yaml


path: /content/dataset_v3
train: images/train
val: images/val

names:
  0: user
  1: internet
  2: firewall
  3: waf
  4: load_balancer
  5: api_gateway
  6: web_server
  7: application_server
  8: microservice
  9: database
  10: cache
  11: queue
  12: object_storage
  13: identity_provider
  14: monitoring


## Treinamento Inicial do Modelo

Nesta etapa é realizado um treinamento curto de 5 épocas utilizando o modelo **YOLOv8n**. O objetivo é validar a configuração do ambiente, verificar se o dataset foi carregado corretamente e confirmar que o treinamento ocorre sem erros antes da execução do treinamento completo.

In [11]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

results = model.train(
    data="/content/dataset_v3/dataset_colab.yaml",
    epochs=5,
    imgsz=640,
    batch=16,
    workers=2,
    cache=False,
    project="/content/runs",
    name="fiap_test"
)

Ultralytics 8.4.83 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset_v3/dataset_colab.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=5, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=fiap_test-2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=

## Resultado do Treinamento Inicial

O treinamento de validação foi concluído com sucesso após **5 épocas**, confirmando que o ambiente, o dataset e a configuração do modelo estão corretos.

As métricas iniciais apresentaram desempenho promissor (**Precision: 89,5%**, **Recall: 91,7%** e **mAP50: 93,8%**), indicando que o modelo já é capaz de identificar corretamente a maior parte dos componentes de arquitetura. Algumas classes, como **Firewall**, ainda apresentaram desempenho inferior devido à menor quantidade de exemplos no dataset, sendo esperado que essas métricas evoluam com o treinamento completo.

Com essa validação, o projeto está apto para iniciar o treinamento definitivo do modelo.

In [12]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

results = model.train(
    data="/content/dataset_v3/dataset_colab.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    workers=2,
    cache=False,
    project="/content/runs",
    name="fiap_full"
)

Ultralytics 8.4.83 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset_v3/dataset_colab.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=fiap_full, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=T

## Resultado do Treinamento Completo

O treinamento definitivo foi concluído com sucesso após 50 épocas, resultando em um modelo com excelente desempenho na detecção dos componentes de arquitetura de software.

As métricas finais alcançaram Precision: 99,9%, Recall: 99,5%, mAP50: 99,4% e mAP50-95: 99,1%, demonstrando alta capacidade de identificar corretamente os elementos presentes nos diagramas. Todas as classes apresentaram desempenho elevado, incluindo Firewall, que mostrou uma melhora significativa em relação ao treinamento inicial, mesmo sendo uma das classes com menor quantidade de exemplos no dataset.

Com a conclusão desta etapa, o modelo best.pt está pronto para ser integrado ao projeto principal, dando início ao desenvolvimento do módulo de detecção automática de componentes que servirá de base para a análise de ameaças utilizando a metodologia STRIDE.

In [13]:
from google.colab import files

files.download("/content/runs/fiap_full/weights/best.pt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>